<a href="https://colab.research.google.com/github/amitkumar15x/flyrank/blob/main/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amitkumar15x/flyrank/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

# Two Paper Findings + My Methodology Questions

## Finding 1: Machine learning can identify content that should be refreshed.

The FlyRank research paper suggests that machine learning can help identify content likely to decline so that teams can prioritize refreshes.

### My methodology question

Where does the target label come from?

If the label is based on future traffic or search performance, it is important to explain exactly how it was created. A clear description of the label generation process helps readers understand whether the model predicts a real outcome or a proxy label.

---

## Finding 2: The proposed model improves prediction quality.

The paper reports improved prediction performance after feature engineering.

### My methodology question

Does the validation strategy support this conclusion?

If pages from the same client appear in both the training and testing sets, the reported performance may be optimistic. A grouped or time-aware validation provides stronger evidence that the model generalizes to unseen data.

These questions are intended to improve transparency and reproducibility rather than criticize the research.

# Honest Validation

In Week 5, I evaluated my model using a stratified 80/20 train-test split.

For this validation audit, I use a grouped validation based on `client_id`. This prevents pages from the same client appearing in both the training and testing data.

This evaluation better represents how the model would perform on completely unseen clients and provides a more realistic estimate of generalization.

In [ ]:
!git clone https://github.com/amitkumar15x/flyrank.git

fatal: destination path 'flyrank' already exists and is not an empty directory.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

In [ ]:
import os

for root, dirs, files in os.walk("/content"):
    for file in files:
        if "content_refresh" in file:
            print(os.path.join(root, file))

/content/flyrank/data/raw/content_refresh_anonymized.csv


In [ ]:
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

df = pd.read_csv("/content/flyrank/data/raw/content_refresh_anonymized.csv")

TARGET = "trend_direction"

groups = df["client_id"]

X = df.drop(columns=[
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id"
])

y = df[TARGET]

X = pd.get_dummies(X, drop_first=True)

X = X.fillna(X.median(numeric_only=True))
X = X.fillna(0)

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Grouped Validation Accuracy:",
      accuracy_score(y_test,pred))

print("Grouped Validation F1:",
      f1_score(y_test,pred,average="weighted"))

Grouped Validation Accuracy: 0.7217264319325004
Grouped Validation F1: 0.6914830562801324


In [ ]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print("Weighted F1:", f1_score(y_test, pred, average="weighted"))

print(classification_report(y_test, pred))

Accuracy: 0.7217264319325004
Weighted F1: 0.6914830562801324
              precision    recall  f1-score   support

        down       0.71      0.94      0.81      3149
        flat       1.00      1.00      1.00       382
         new       1.00      1.00      1.00       302
      stable       0.52      0.32      0.40      1301
          up       0.78      0.37      0.50      1029

    accuracy                           0.72      6163
   macro avg       0.80      0.73      0.74      6163
weighted avg       0.71      0.72      0.69      6163



# Honest Validation

In Week 5, I evaluated my model using a stratified 80/20 train-test split.

For this validation audit, I re-ran the model using a grouped split based on `client_id`. This prevents pages from the same client appearing in both the training and testing sets, providing a more realistic estimate of how the model performs on unseen clients.

## Comparison

| Validation Method | Accuracy | Weighted F1 |
|-------------------|---------:|------------:|
| Week 5 Stratified Split | 0.789 | 0.771 |
| Week 6 Grouped Split | 0.722 | 0.691 |

The grouped validation produced lower performance than the original stratified split. This suggests the original evaluation may have benefited from similarities between training and testing data. The grouped split provides a more conservative estimate of generalization and better reflects deployment on new clients.

# Leakage Audit

I reviewed my final feature set to identify possible sources of target leakage.

Before training the model, I removed the following columns:

- trend_direction (target variable)
- trend_pct (future information)
- content_id (identifier)
- client_id (used only for grouped validation)

The remaining features represent historical observations that would reasonably be available at prediction time. Based on this review, I did not identify obvious target leakage in the final feature set.

# Claim Rewrite

## Original Claim

The Random Forest model accurately predicts future content performance.

## Revised Claim

In this dataset, the Random Forest model achieved useful predictive performance under the evaluated validation strategy. The grouped validation showed lower accuracy than the original stratified split, indicating that model performance is sensitive to the validation design. These results should be interpreted as decision-support rather than proof of future performance.

# Error Analysis

The confusion matrix and classification report show that the model performed best on the **down**, **flat**, and **new** classes. These classes achieved high precision and recall.

Most errors occurred between the **stable** and **up** categories. These classes had lower recall, indicating that the model sometimes confused pages with similar performance patterns.

The grouped validation produced an overall accuracy of **72.17%** and a weighted F1 score of **0.691**. Compared with the Week 5 stratified split, this decrease suggests that grouped validation provides a more realistic estimate of model performance on unseen clients.

# Self Check

- ✅ Every notebook section has been completed.
- ✅ The notebook runs from top to bottom without errors.
- ✅ Two findings from the research paper were reviewed.
- ✅ A grouped validation strategy was used.
- ✅ Before and after validation results were compared.
- ✅ A leakage audit was completed.
- ✅ Claims were rewritten using careful language such as "observed", "measured", "directional", and "decision-support".
- ✅ No private client information or confidential data was included.